In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [ ]:
##Loading the dataset
data = pd.read_csv('Churn_Modelling.csv')
data.head()

In [ ]:
##Preprocessing the data
##Dropping irrelevant features
data = data.drop(['RowNumber','CustomerId','Surname'], axis = 1)
data

In [ ]:
##Applying encoding for categorical values (Geography, Gender)
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
data

In [ ]:
##One Hot encoding Geography Column
onehot_encoder_geo = OneHotEncoder(sparse_output= False)
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']])
geo_df = pd.DataFrame(geo_encoded, columns = onehot_encoder_geo.get_feature_names_out(['Geography']))
data = data.drop('Geography',axis=1)
data = pd.concat([data,geo_df],axis=1)
data

In [ ]:
##saving the encoders and the scalars in a pickle
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file: 
    pickle.dump(onehot_encoder_geo,file)
    

In [ ]:
##Divide the data into dependant and independant features
X = data.drop('Exited',axis=1)
Y  = data['Exited']

##Splitting into training and testing data
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size = 0.2, random_state=42)

##Scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

ANN Implementation 

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [ ]:
##Building ANN Model
model = Sequential([
    Dense(64,activation='relu',input_shape = (X_train.shape[1],)),##1ST HIDEEN LAYER
    Dense(32,activation='relu'), ##HL 2
    Dense(1,activation='sigmoid') ##OUTPUT
])

In [ ]:
model.summary()

In [ ]:
opt = tf.keras.optimizers.Adam(learning_rate=0.01)

In [ ]:
##Compiple the model
model.compile(opt,loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
## Set up tensorboard
log_dir = "logs/fit" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir= log_dir, histogram_freq=1)

In [ ]:
##Set up early stopping
early_stopping_callback = EarlyStopping(monitor='val_loss',patience = 5, restore_best_weights=True)

In [ ]:
##Training the model
history = model.fit(
    X_train, Y_train,validation_data = (X_test,Y_test),epochs = 100,callbacks = [tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.save('model.h5')

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir logs